# Sanity Check
Testen, ob alle m6A modification positionen auch A sind

In [3]:
import gzip
import re
import csv
from src.util import load_region_modifications
from src.files.files import get_files

def sanity_check(fastaFile, regions):
    with fastaFile.open_or_recompute() as fastaFile:
        is_in_region = False
        has_region_modifications = False
        current_region_name = None
        current_region_modifications = None
        
        for line in fastaFile:
            if re.search(">", line):
                first_colon = line.find(":")
                
                is_in_region = True
                current_region_name = line[1:first_colon]
                current_region_modifications = []
                has_region_modifications = False
                if current_region_name in regions:
                    current_region_modifications = regions[current_region_name].modifications
                    if len(current_region_modifications):
                        has_region_modifications = True
            else:
                if not is_in_region:
                    continue
                if not has_region_modifications:
                    continue
                line_length = len(line)

                for modification in current_region_modifications:
                    if modification > line_length:
                        break

                    char = line[modification]

                    if char != "A" and char != "a":
                        print("Non A m6A!")

                current_region_modifications = filter(lambda modification: modification > line_length, current_region_modifications)
                current_region_modifications = [x - line_length for x in current_region_modifications]
                if len(current_region_modifications) == 0:
                    has_region_modifications = False

for key, file in get_files().get_assembled_region_local_intersects_assembled_files().get_files_dict().items():
    print("Checking", key)
    regions = load_region_modifications(file)

    sanity_check(get_files().get_assembled_region_fasta_files().get_files_dict()[key], regions)

print("Done")

Checking 3utr
Checking 5utr
Checking 5utr_start
Checking cds
Checking coding_exons
Checking coding_introns
Checking exons
Checking introns
Checking non_coding_exons
Checking non_coding_introns
Done
